# Gestión de excepciones

Por muy bien que escribamos nuestro código, **las cosas fallan**: un usuario
introduce un texto donde esperábamos un número, un archivo no existe, una conexión se
cae, una división acaba entre cero... Cuando ocurre algo así, Python lanza una
**excepción**, y si no hacemos nada, el programa **se detiene** de golpe.

Gestionar excepciones consiste en **anticipar esos fallos y decidir qué hacer** cuando
ocurren, en lugar de dejar que el programa reviente. En este vídeo veremos:

- Qué es una excepción y cómo leer un *traceback*.
- `try` / `except` para capturar errores, y por qué hay que ser **específico**.
- `else` y `finally`.
- `raise` para lanzar nuestras propias excepciones.
- Crear **excepciones personalizadas**.
- Buenas prácticas (encadenar excepciones, el estilo *EAFP*...).

> 📚 Documentación oficial de referencia:
> **[Errores y excepciones](https://docs.python.org/es/3/tutorial/errors.html)**.

---

## 1. ¿Qué es una excepción? El *traceback*

Cuando Python encuentra un error que no puede resolver, **lanza** (*raise*) una
excepción. Si nadie la captura, el programa se para y muestra un **traceback**: un
informe de qué pasó y dónde.

Por ejemplo, este código:

```python
numeros = [1, 2, 3]
print(numeros[10])
```

produce un traceback como este:

```
Traceback (most recent call last):
  File "programa.py", line 2, in <module>
    print(numeros[10])
          ~~~~~~~^^^^
IndexError: list index out of range
```

Leerlo de **abajo arriba** es la clave:

- La **última línea** es lo más importante: el **tipo** de excepción (`IndexError`) y
  el **mensaje** (`list index out of range`).
- Las líneas de arriba indican **dónde** ocurrió (archivo, línea y código).

Cada tipo de error tiene su propia clase de excepción. Estos son algunos muy comunes:

| Excepción | Cuándo ocurre |
|-----------|----------------|
| `ValueError` | Un valor del tipo correcto pero inadecuado (`int("hola")`) |
| `TypeError` | Una operación entre tipos incompatibles (`"2" + 2`) |
| `KeyError` | Una clave que no existe en un diccionario |
| `IndexError` | Un índice fuera del rango de una lista |
| `ZeroDivisionError` | Una división entre cero |
| `FileNotFoundError` | Un archivo que no existe |

---

## 2. `try` / `except`: capturar el error

Para que un error **no detenga** el programa, envolvemos el código que puede fallar en
un bloque `try`, y ponemos en `except` lo que queremos hacer si ocurre la excepción:

In [ ]:
try:
    resultado = 10 / 0
    print(resultado)  # esta línea no llega a ejecutarse
except ZeroDivisionError:
    print("No se puede dividir entre cero")

print("El programa continúa con normalidad")

El flujo es: Python ejecuta el `try`; si salta la excepción indicada, **abandona el
`try`** y ejecuta el `except`. Como hemos gestionado el error, el programa **sigue**
en vez de pararse.

---

## 3. Ser específico: capturar el tipo correcto

Es **muy importante** capturar solo las excepciones que esperamos. Un error frecuente
es usar un `except` "a secas" (sin tipo), que atrapa **absolutamente todo**, incluso
errores que ni habíamos imaginado y que quizá deberían detener el programa.

```python
# ⚠️ MAL: esconde cualquier error, hasta los que no esperabas
try:
    procesar()
except:
    pass  # silencio total: si algo va mal, nunca te enterarás
```

Lo correcto es nombrar el tipo de excepción. Y podemos acceder al objeto de la
excepción con `as` para leer su mensaje:

In [ ]:
def convertir_a_entero(valor):
    try:
        return int(valor)
    except ValueError as e:
        print(f"No se pudo convertir: {e}")
        return None


print(convertir_a_entero("123"))   # 123
print(convertir_a_entero("hola"))  # error controlado -> None

Podemos manejar **varios tipos** de excepción, ya sea con bloques `except` separados
(para dar una respuesta distinta a cada uno) o agrupándolos en una tupla:

In [ ]:
def acceder(coleccion, clave):
    try:
        return coleccion[clave]
    except KeyError:
        return "(esa clave no existe)"
    except IndexError:
        return "(ese índice está fuera de rango)"


print(acceder({"nombre": "Ana"}, "edad"))  # KeyError -> mensaje
print(acceder([1, 2, 3], 10))               # IndexError -> mensaje

In [ ]:
# Varios tipos con la MISMA respuesta: una tupla en un solo except
def convertir(valor):
    try:
        return int(valor)
    except (ValueError, TypeError) as e:
        print(f"No se pudo convertir ({type(e).__name__}): {e}")
        return None


print(convertir("42"))
print(convertir("hola"))  # ValueError
print(convertir(None))    # TypeError

---

## 4. `else` y `finally`

El `try` admite dos bloques opcionales más:

- **`else`**: se ejecuta **solo si NO hubo excepción**. Sirve para poner ahí el código
  que depende de que el `try` haya ido bien.
- **`finally`**: se ejecuta **siempre**, haya error o no. Es el sitio para tareas de
  limpieza (cerrar archivos, liberar recursos...).

In [ ]:
def leer_numero(texto):
    try:
        numero = int(texto)
    except ValueError:
        print(f"  '{texto}' no es un número válido")
    else:
        print(f"  Número válido: {numero}")  # solo si el try fue bien
    finally:
        print("  (fin del intento)")  # siempre, pase lo que pase


print("leer_numero('42'):")
leer_numero("42")
print("leer_numero('abc'):")
leer_numero("abc")

> 💡 En la práctica, para liberar recursos casi siempre es más cómodo y seguro usar un
> gestor de contexto (`with`) que un `finally` a mano. Pero entender `finally` ayuda a
> ver qué hace `with` por debajo.

---

## 5. `raise`: lanzar nuestras propias excepciones

No solo capturamos excepciones: también podemos **lanzarlas** nosotros con `raise`
cuando detectamos una situación inválida. Es la forma correcta de decir "esto no
debería pasar", en lugar de devolver un valor raro o seguir como si nada.

In [ ]:
def raiz_cuadrada(x):
    if x < 0:
        raise ValueError("No existe raíz cuadrada real de un número negativo")
    return x**0.5


print(raiz_cuadrada(9))  # 3.0

try:
    raiz_cuadrada(-1)
except ValueError as e:
    print("Error capturado:", e)

> 💡 Lanzar una excepción con un **mensaje claro** es mucho mejor que devolver `None` o
> `-1` para indicar un error: obliga a quien usa tu función a enterarse del problema en
> vez de arrastrar un valor inválido sin darse cuenta.

---

## 6. Excepciones personalizadas

Para errores propios de **nuestro dominio**, lo mejor es crear nuestra **propia clase**
de excepción. Basta con heredar de `Exception`. Así, quien use nuestro código puede
capturar **exactamente** ese error.

Retomamos la cuenta bancaria del vídeo de testing: retirar más dinero del disponible
no es un `ValueError` genérico, es un `SaldoInsuficiente`.

In [ ]:
class SaldoInsuficiente(Exception):
    """Se lanza al intentar retirar más dinero del disponible."""


class Cuenta:
    def __init__(self, saldo=0):
        self.saldo = saldo

    def retirar(self, cantidad):
        if cantidad > self.saldo:
            raise SaldoInsuficiente(
                f"Intentas retirar {cantidad}€ pero solo tienes {self.saldo}€"
            )
        self.saldo -= cantidad


cuenta = Cuenta(100)
try:
    cuenta.retirar(150)
except SaldoInsuficiente as e:
    print("Operación rechazada:", e)

Una excepción personalizada puede llevar **datos extra** en su `__init__`, útiles para
que quien la capture pueda reaccionar con detalle:

In [ ]:
class SaldoInsuficiente(Exception):
    def __init__(self, saldo, solicitado):
        self.saldo = saldo
        self.solicitado = solicitado
        self.faltan = solicitado - saldo
        super().__init__(f"Faltan {self.faltan}€ para esta operación")


try:
    raise SaldoInsuficiente(saldo=100, solicitado=150)
except SaldoInsuficiente as e:
    print("Mensaje:", e)
    print("Te faltan:", e.faltan, "€")  # accedemos a los datos extra

---

## 7. Encadenar excepciones con `raise ... from`

A veces capturamos un error técnico de bajo nivel y queremos relanzarlo como un error
más comprensible de nuestro dominio, **sin perder** la causa original. Para eso está
`raise NuevaExcepcion(...) from error_original`:

In [ ]:
class ErrorDeConfiguracion(Exception):
    pass


def cargar_puerto(texto):
    try:
        return int(texto)
    except ValueError as e:
        # relanzamos con un mensaje de nuestro dominio, conservando la causa
        raise ErrorDeConfiguracion(f"Puerto inválido: {texto!r}") from e


try:
    cargar_puerto("ABC")
except ErrorDeConfiguracion as e:
    print("Error:", e)
    print("Causado por:", repr(e.__cause__))  # la excepción original sigue ahí

> 💡 En el *traceback* real verías el mensaje *"The above exception was the direct
> cause of the following exception"*, que conecta ambos errores. Esto facilita
> muchísimo la depuración: ves el error "bonito" y, debajo, qué lo provocó.

---

## 8. Buenas prácticas

- **Sé específico**: captura el tipo de excepción que esperas, no un `except:` genérico
  que se traga todo (incluidos errores de programación que querrías ver).
- **No silencies errores**: un `except: pass` esconde problemas. Como mínimo, registra
  o informa de lo que pasó.
- **Captura cerca de donde puedas hacer algo útil**: no envuelvas todo en un `try`
  gigante; rodea solo la operación que puede fallar.
- **Mensajes claros** al lanzar (`raise`): que digan qué pasó y, si puedes, cómo
  arreglarlo.

### EAFP: "pedir perdón, no permiso"

En Python es muy común el estilo **EAFP** (*Easier to Ask Forgiveness than
Permission*): en lugar de comprobar mil condiciones antes de actuar (estilo **LBYL**,
*Look Before You Leap*), intentamos la operación y capturamos el error si ocurre.

In [ ]:
datos = {"nombre": "Ana"}

# LBYL: comprobar antes (más verboso)
if "edad" in datos:
    print("LBYL:", datos["edad"])
else:
    print("LBYL: no hay edad")

# EAFP: intentar y capturar (más pythonico)
try:
    print("EAFP:", datos["edad"])
except KeyError:
    print("EAFP: no hay edad")

---

## 9. Resumen

En este vídeo hemos aprendido a gestionar los errores en lugar de sufrirlos:

1. Una **excepción** se lanza cuando algo falla; sin gestionarla, detiene el programa
   y muestra un **traceback** (que se lee de abajo arriba).
2. **`try` / `except`** captura la excepción y permite que el programa continúe.
3. Hay que **ser específico** con el tipo capturado y evitar el `except:` que lo traga
   todo.
4. **`else`** se ejecuta si no hubo error; **`finally`** se ejecuta siempre.
5. **`raise`** lanza excepciones propias ante situaciones inválidas.
6. Podemos crear **excepciones personalizadas** heredando de `Exception`, con datos
   extra si hace falta.
7. **`raise ... from`** encadena excepciones conservando la causa original.
8. Estilo **EAFP**: intentar y capturar suele ser más pythonico que comprobarlo todo
   de antemano.

| Elemento | Para qué |
|----------|----------|
| `try` | Envolver el código que puede fallar |
| `except TipoError as e` | Capturar un tipo concreto y leer el error |
| `else` | Código que solo corre si NO hubo excepción |
| `finally` | Código que corre SIEMPRE (limpieza) |
| `raise` | Lanzar una excepción |
| `class MiError(Exception)` | Crear una excepción propia |
| `raise ... from e` | Encadenar excepciones conservando la causa |